In [1]:
# import socket
#
# def test_network_connection(ip_address, port=5025):
#     """Test if BNC 765 is reachable on the network"""
#     print(f"Testing connection to {ip_address}:{port}...")
#
#     try:
#         # Try to ping the device
#         sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
#         sock.settimeout(2)
#         result = sock.connect_ex((ip_address, port))
#         sock.close()
#
#         if result == 0:
#             print(f"✓ Port {port} is OPEN on {ip_address}")
#             print(f"  Device appears to be reachable!")
#             return True
#         else:
#             print(f"✗ Port {port} is CLOSED on {ip_address}")
#             print(f"  Device may be off or on different network")
#             return False
#
#     except Exception as e:
#         print(f"✗ Network error: {e}")
#         return False
#
#
# # Test it
# # BNC_IP = '169.254.209.156'  # Kepler pulser
# BNC_IP = '169.254.125.69'  # HMMB pulser
# test_network_connection(BNC_IP)

Testing connection to 169.254.125.69:5025...
✗ Port 5025 is CLOSED on 169.254.125.69
  Device may be off or on different network


False

In [1]:
import socket
import time

def scan_instrument_ports(ip_address):
    """Scan common instrument control ports"""

    print("="*70)
    print(f"SCANNING PORTS ON {ip_address}")
    print("="*70 + "\n")

    # Common instrument ports to check
    ports_to_check = [
        (5025, "SCPI Raw Socket (most common)"),
        (5024, "SCPI Raw Socket (alt)"),
        (111, "VXI-11"),
        (5000, "Custom SCPI"),
        (5001, "Custom SCPI"),
        (5002, "Custom SCPI"),
        (9221, "LXI"),
        (23, "Telnet"),
        (80, "HTTP/Web interface"),
        (10001, "Custom"),
        (10002, "Custom"),
    ]

    open_ports = []

    for port, description in ports_to_check:
        print(f"Testing port {port:5d} ({description})...", end=" ")

        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex((ip_address, port))
            sock.close()

            if result == 0:
                print("✓ OPEN")
                open_ports.append((port, description))
            else:
                print("✗ closed")

        except Exception as e:
            print(f"✗ error: {e}")

    print("\n" + "="*70)
    print("OPEN PORTS FOUND:")
    print("="*70)

    if open_ports:
        for port, desc in open_ports:
            print(f"  Port {port}: {desc}")
    else:
        print("  No open ports found!")
        print("\nTroubleshooting:")
        print("  1. Verify IP address is correct")
        print("  2. Check device is powered on")
        print("  3. Check network cable connected")
        print("  4. Try ping command")
        print("  5. Check firewall settings")

    print()
    return open_ports


# Scan for open ports

# BNC_IP = '169.254.209.156'  # Kepler pulser
BNC_IP = '169.254.125.69'  # HMMB pulser

open_ports = scan_instrument_ports(BNC_IP)

SCANNING PORTS ON 169.254.125.69

Testing port  5025 (SCPI Raw Socket (most common))... ✗ closed
Testing port  5024 (SCPI Raw Socket (alt))... ✗ closed
Testing port   111 (VXI-11)... ✓ OPEN
Testing port  5000 (Custom SCPI)... ✗ closed
Testing port  5001 (Custom SCPI)... ✗ closed
Testing port  5002 (Custom SCPI)... ✗ closed
Testing port  9221 (LXI)... ✗ closed
Testing port    23 (Telnet)... ✗ closed
Testing port    80 (HTTP/Web interface)... ✗ closed
Testing port 10001 (Custom)... ✗ closed
Testing port 10002 (Custom)... ✗ closed

OPEN PORTS FOUND:
  Port 111: VXI-11



In [ ]:
import pyvisa
import time

class BNC765Controller:
    """
    Controller for BNC 765 / Active Technologies AT-PULSE-RIDER-PG1074

    Note: This device only supports querying WIDTH and DELAY.
    Other parameters (period, amplitude, polarity) can be SET but not READ.
    """

    def __init__(self, ip_address='169.254.209.156'):
        """Connect to pulse generator"""
        self.rm = pyvisa.ResourceManager()
        address = f'TCPIP::{ip_address}::INSTR'
        self.pulse_gen = self.rm.open_resource(address)
        self.pulse_gen.timeout = 10000

        idn = self.pulse_gen.query('*IDN?')
        print(f"Connected to: {idn.strip()}")
        self.pulse_gen.write('*CLS')

        # Store settings locally since we can't query them
        self.channel_settings = {
            1: {}, 2: {}, 3: {}, 4: {}
        }

    def setup_channel(self, channel=1, width=100e-9, period=1e-6,
                     amplitude=2., delay=0, polarity='NORM'):
        """
        Set up a pulse channel

        Args:
            channel: Channel number (1-4)
            width: Pulse width in seconds
            period: Pulse period in seconds (or use frequency)
            amplitude: Pulse amplitude in volts
            delay: Pulse delay in seconds
            polarity: 'NORM' (positive) or 'INV' (negative/inverted)
        """
        print(f"\nConfiguring Channel {channel}:")

        # Disable channel first
        self.disable_channel(channel)
        time.sleep(0.1)

        # Set WIDTH
        self.pulse_gen.write(f':PULSE{channel}:WIDTH {width}')
        actual_width = float(self.pulse_gen.query(f':PULSE{channel}:WIDTH?').strip())
        print(f"  Width:     {actual_width*1e9:.1f} ns")

        # Set PERIOD (can write but not read back)
        self.pulse_gen.write(f':PULSE{channel}:PERIOD {period}')
        print(f"  Period:    {period*1e6:.3f} μs ({1/period/1e6:.3f} MHz) [set, cannot verify]")

        # Set AMPLITUDE (can write but not read back)
        self.pulse_gen.write(f':PULSE{channel}:AMPLITUDE {amplitude}')
        print(f"  Amplitude: {amplitude} V [set, cannot verify]")

        # Set DELAY
        self.pulse_gen.write(f':PULSE{channel}:DELAY {delay}')
        actual_delay = float(self.pulse_gen.query(f':PULSE{channel}:DELAY?').strip())
        print(f"  Delay:     {actual_delay*1e9:.1f} ns")

        # Set POLARITY (can write but not read back)
        self.pulse_gen.write(f':PULSE{channel}:POLARITY {polarity}')
        polarity_str = "UP/Positive" if polarity == 'NORM' else "DOWN/Negative"
        print(f"  Polarity:  {polarity} ({polarity_str}) [set, cannot verify]")

        # Store settings locally
        self.channel_settings[channel] = {
            'width': width,
            'period': period,
            'amplitude': amplitude,
            'delay': delay,
            'polarity': polarity
        }

        print(f"  ✓ Channel {channel} configured")

    def enable_channel(self, channel=1):
        """Enable channel output"""
        self.pulse_gen.write(f':OUTPUT{channel} ON')

        # Verify
        state = int(self.pulse_gen.query(f':OUTPUT{channel}?').strip())
        if state == 1:
            print(f"✓ CH{channel} output ENABLED")
        else:
            print(f"⚠ CH{channel} state unclear (sent enable command)")

    def disable_channel(self, channel=1):
        """Disable channel output"""
        self.pulse_gen.write(f':OUTPUT{channel} OFF')

        # Verify
        state = int(self.pulse_gen.query(f':OUTPUT{channel}?').strip())
        if state == 0:
            print(f"✓ CH{channel} output DISABLED")
        else:
            print(f"⚠ CH{channel} state unclear (sent disable command)")

    def enable_all_channels(self, channels=[1, 2, 3, 4]):
        """Enable multiple channels"""
        print(f"\nEnabling channels {channels}...")
        for ch in channels:
            self.pulse_gen.write(f':OUTPUT{ch} ON')
        print(f"✓ All channels enabled")

    def disable_all_channels(self, channels=[1, 2, 3, 4]):
        """Disable multiple channels"""
        print(f"\nDisabling channels {channels}...")
        for ch in channels:
            self.pulse_gen.write(f':OUTPUT{ch} OFF')
        print(f"✓ All channels disabled")

    def get_channel_state(self, channel=1):
        """Get channel output state"""
        state = int(self.pulse_gen.query(f':OUTPUT{channel}?').strip())
        return state == 1

    def get_channel_settings(self, channel=1):
        """
        Get channel settings
        Note: Returns stored values for parameters that can't be queried
        """
        print(f"\nChannel {channel} Settings:")

        # Query what we can
        try:
            width = float(self.pulse_gen.query(f':PULSE{channel}:WIDTH?').strip())
            print(f"  Width:     {width*1e9:.1f} ns [verified]")
        except:
            width = None
            print(f"  Width:     (query failed)")

        try:
            delay = float(self.pulse_gen.query(f':PULSE{channel}:DELAY?').strip())
            print(f"  Delay:     {delay*1e9:.1f} ns [verified]")
        except:
            delay = None
            print(f"  Delay:     (query failed)")

        try:
            state = int(self.pulse_gen.query(f':OUTPUT{channel}?').strip())
            state_str = "ON" if state == 1 else "OFF"
            print(f"  Output:    {state_str} [verified]")
        except:
            print(f"  Output:    (query failed)")

        # Show stored values for parameters we can't query
        if channel in self.channel_settings and self.channel_settings[channel]:
            settings = self.channel_settings[channel]
            print(f"\n  Stored settings (cannot verify from device):")
            if 'period' in settings:
                print(f"    Period:    {settings['period']*1e6:.3f} μs ({1/settings['period']/1e6:.3f} MHz)")
            if 'amplitude' in settings:
                print(f"    Amplitude: {settings['amplitude']} V")
            if 'polarity' in settings:
                pol_str = "UP/Positive" if settings['polarity'] == 'NORM' else "DOWN/Negative"
                print(f"    Polarity:  {settings['polarity']} ({pol_str})")

        return {
            'width': width,
            'delay': delay,
            'stored_settings': self.channel_settings.get(channel, {})
        }

    def close(self):
        """Close connection and disable all outputs"""
        print("\nClosing connection...")
        self.disable_all_channels()
        self.pulse_gen.close()
        self.rm.close()
        print("✓ Connection closed, all outputs disabled")


# ============================================================================
# COMPLETE TEST
# ============================================================================

if __name__ == '__main__':

    print("="*70)
    print("BNC 765 PULSE GENERATOR - COMPLETE TEST")
    print("="*70)

    # bnc = BNC765Controller('169.254.209.156')
    bnc = BNC765Controller('169.254.125.69') #HMMB pulser

    try:
        # TEST 1: Single channel
        print("\n" + "="*70)
        print("TEST 1: Single Channel Pulse")
        print("="*70)

        bnc.setup_channel(
            channel=1,
            width=100e-9,      # 100 ns
            period=1e-6,       # 1 MHz
            amplitude=2.,     # 2.V
            delay=0,
            polarity='NORM'    # Positive/UP
        )

        bnc.get_channel_settings(channel=1)

        input("\nPress Enter to ENABLE CH1 (connect scope!)...")
        bnc.enable_channel(1)

        print("\n>> You should see 2.V positive pulses at 1 MHz on CH1 <<")
        print("   Width: 100 ns")

        input("\nPress Enter to DISABLE CH1...")
        bnc.disable_channel(1)

        # TEST 2: 4-Channel Pattern
        print("\n" + "="*70)
        print("TEST 2: 4-Channel Mixed Polarity Pattern")
        print("="*70)
        print("\nPattern: DOWN, UP, DOWN, DOWN")
        print("Timing:  t=0, t=500ns, t=1μs, t=1.5μs")

        input("\nPress Enter to configure 4-channel pattern...")

        pulse_width = 100e-9        # 100 ns pulse width
        pulse_spacing = 500e-9      # 500 ns between pulses
        pattern_period = 10e-6      # Pattern repeats every 10 μs
        amplitude = 2.0            # 2.0V

        # CH1: DOWN pulse at t=0
        bnc.setup_channel(
            channel=1,
            width=pulse_width,
            period=pattern_period,
            amplitude=amplitude,
            delay=0,
            polarity='INV'          # INV = DOWN/Negative
        )

        # CH2: UP pulse at t=500ns
        bnc.setup_channel(
            channel=2,
            width=pulse_width,
            period=pattern_period,
            amplitude=amplitude,
            delay=pulse_spacing,
            polarity='NORM'         # NORM = UP/Positive
        )

        # CH3: DOWN pulse at t=1μs
        bnc.setup_channel(
            channel=3,
            width=pulse_width,
            period=pattern_period,
            amplitude=amplitude,
            delay=2*pulse_spacing,
            polarity='INV'          # DOWN
        )

        # CH4: DOWN pulse at t=1.5μs
        bnc.setup_channel(
            channel=4,
            width=pulse_width,
            period=pattern_period,
            amplitude=amplitude,
            delay=3*pulse_spacing,
            polarity='INV'          # DOWN
        )

        print("\n✓ All 4 channels configured!")
        print("\nHardware setup:")
        print("  1. Connect CH1, CH2, CH3, CH4 outputs to resistor summing network")
        print("  2. Connect summed output to oscilloscope CH1")
        print("  3. Scope trigger: CH1, FALLING edge, ~1.65V level")

        input("\nPress Enter to ENABLE all 4 channels...")
        bnc.enable_all_channels([1, 2, 3, 4])

        print("\n>> Check oscilloscope <<")
        print("   You should see 4-pulse pattern repeating every 10 μs:")
        print("   ↓ (down) at t=0")
        print("   ↑ (up) at t=500ns")
        print("   ↓ (down) at t=1.0μs")
        print("   ↓ (down) at t=1.5μs")

        input("\nPress Enter to DISABLE all outputs...")
        bnc.disable_all_channels([1, 2, 3, 4])

        print("\n✓ All tests complete!")

    except KeyboardInterrupt:
        print("\n\nInterrupted by user")
    except Exception as e:
        print(f"\n✗ Error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        bnc.close()
        print("\n" + "="*70)
        print("✓ BNC 765 test complete!")
        print("="*70)

BNC 765 PULSE GENERATOR - COMPLETE TEST
Connected to: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1072,00000001,SCPI:99.0,SV:1.0.0.0

TEST 1: Single Channel Pulse

Configuring Channel 1:
✓ CH1 output DISABLED
  Width:     100.0 ns
  Period:    1.000 μs (1.000 MHz) [set, cannot verify]
  Amplitude: 2.0 V [set, cannot verify]
  Delay:     0.0 ns
  Polarity:  NORM (UP/Positive) [set, cannot verify]
  ✓ Channel 1 configured

Channel 1 Settings:
  Width:     100.0 ns [verified]
  Delay:     0.0 ns [verified]
  Output:    OFF [verified]

  Stored settings (cannot verify from device):
    Period:    1.000 μs (1.000 MHz)
    Amplitude: 2.0 V
    Polarity:  NORM (UP/Positive)


In [ ]:
import pyvisa
import time

BNC_IP = '169.254.125.69'

rm = pyvisa.ResourceManager()
pulser = rm.open_resource(f'TCPIP::{BNC_IP}::INSTR')
pulser.timeout = 10000

# Setup
pulser.write(':OUTPUT1 OFF')
pulser.write('SOURCE1:PULSE:WIDTH 200E-9')
pulser.write('SOURCE1:PULSE:PERIOD 1000')  # Won't repeat for 1000 seconds
pulser.write('SOURCE1:VOLTAGE:AMPLITUDE 2.0')
pulser.write(':PULSE1:DELAY 0')
pulser.write(':PULSE1:POLARITY NORM')

print("Ready to fire single pulse")
input("Press Enter...")

# Fire
pulser.write(':OUTPUT1 ON')
time.sleep(0.001)
pulser.write(':OUTPUT1 OFF')

print("✓ Pulse fired!")

pulser.close()
rm.close()

SINGLE-SHOT 4-PULSE PATTERN GENERATOR

This configures the BNC 765 to fire a pattern ONCE

✓ Connected to: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1072,00000001,SCPI:99.0,SV:1.0.0.0

Checking for burst mode capability...
  → Burst mode commands accepted

CONFIGURING SINGLE-SHOT 4-PULSE PATTERN

Pattern Timeline (fires ONCE):
  Pulse 1: t=   0- 200ns   +0.2V
           (wait 200ns)
  Pulse 2: t= 400- 600ns   -0.2V
           (wait 800ns)
  Pulse 3: t=1400-1600ns   +0.2V
           (wait 300ns)
  Pulse 4: t=1900-2100ns   +0.2V

Total duration: 2.100 μs
Then: STOPS (no repeat)

Channel Allocation:
  CH1 (Positive): 3 pulse(s)
  CH2 (Negative): 1 pulse(s)


----------------------------------------------------------------------
CH1 CONFIGURATION (NORM polarity)
----------------------------------------------------------------------
  Multiple pulses:
    Pulse #1: t=0ns
    Pulse #3: t=1400ns
    Pulse #4: t=1900ns
    Period: 2100ns
    ⚠ Will use timed enable/disable for single-shot
  Verifie

In [40]:
import pyvisa
import time

# Configuration
BNC_IP = '169.254.125.69'
CHANNEL = 1

# Connect to BNC 765
rm = pyvisa.ResourceManager()
pulser = rm.open_resource(f'TCPIP::{BNC_IP}::INSTR')
pulser.timeout = 10000

print("Connected to:", pulser.query('*IDN?').strip())

# Configure for single pulse using low frequency
print("\nConfiguring for single pulse...")

# Turn off output initially
pulser.write(f'OUTPUT{CHANNEL}:STATE OFF')
time.sleep(0.1)

# Set pulse width (100 ns)
pulser.write(f'SOURCE{CHANNEL}:PULSE:WIDTH 100E-9')
time.sleep(0.1)

# Set amplitude (1 V)
pulser.write(f'SOURCE{CHANNEL}:VOLTAGE:LEVEL 1.0')
time.sleep(0.1)

# Set very low frequency for single pulse (e.g., 1 Hz = 1 pulse per second)
pulser.write(f'SOURCE{CHANNEL}:FREQUENCY 1')  # 1 Hz
time.sleep(0.1)

# Verify settings
width = pulser.query(f'SOURCE{CHANNEL}:PULSE:WIDTH?').strip()
level = pulser.query(f'SOURCE{CHANNEL}:VOLTAGE:LEVEL?').strip()
freq = pulser.query(f'SOURCE{CHANNEL}:FREQUENCY?').strip()
duty = pulser.query(f'SOURCE{CHANNEL}:PULSE:DCYCLE?').strip()

print(f"Width: {float(width)*1e9:.1f} ns")
print(f"Level: {level} V")
print(f"Frequency: {freq} Hz")
print(f"Duty Cycle: {duty}%")

# Check for errors
err = pulser.query('SYST:ERR?').strip()
if 'No error' in err or err == '0':
    print("Configuration OK")
else:
    print(f"Error: {err}")

# Generate single pulse by turning on briefly
input("\nPress Enter to generate single pulse...")

pulser.write(f'OUTPUT{CHANNEL}:STATE ON')
print("Output ON - pulse generating...")
time.sleep(0.15)  # Wait slightly longer than one period (>1/1Hz = 1 second for 1 pulse)
pulser.write(f'OUTPUT{CHANNEL}:STATE OFF')
print("Output OFF - single pulse completed")

# For controlled multiple single pulses
while True:
    response = input("\nGenerate another single pulse? (y/n): ")
    if response.lower() == 'y':
        pulser.write(f'OUTPUT{CHANNEL}:STATE ON')
        time.sleep(0.15)  # One pulse at 1 Hz
        pulser.write(f'OUTPUT{CHANNEL}:STATE OFF')
        print("Pulse generated!")
    else:
        break

# Clean up
pulser.write(f'OUTPUT{CHANNEL}:STATE OFF')
pulser.close()
print("\nConnection closed")

Connected to: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1072,00000001,SCPI:99.0,SV:1.0.0.0

Configuring for single pulse...
Width: 500000000.0 ns
Level: 1 V
Frequency: 1 Hz
Duty Cycle: 50%
Error: Error: 70, Command keywords were not recognized
Output ON - pulse generating...
Output OFF - single pulse completed

Connection closed


In [35]:
rm = pyvisa.ResourceManager()
pulser = rm.open_resource(f'TCPIP::{BNC_IP}::INSTR')
pulser.timeout = 10000

cmd = 'OUTPUT1:STATE OFF'
cmd = 'SOURCE1:PULSE:WIDTH 100E-9'
cmd = 'SOURCE1:VOLTAGE:LEVEL 1.0'

pulser.write(cmd)
time.sleep(0.1)
# err = pulser.query('SYST:ERR?').strip()

In [70]:
import pyvisa
import time

# Configuration
BNC_IP = '169.254.125.69'
CHANNEL = 1

# Connect
rm = pyvisa.ResourceManager()
pulser = rm.open_resource(f'TCPIP::{BNC_IP}::INSTR')
pulser.timeout = 10000

print("Connected to:", pulser.query('*IDN?').strip())

# Configure single pulse in burst mode
print("\nConfiguring burst mode for single pulse (0V → 1V)...")

# Turn off output initially
pulser.write(f'OUTPUT{CHANNEL}:STATE OFF')
time.sleep(0.1)

# Set pulse parameters
pulser.write(f'SOURCE{CHANNEL}:VOLTAGE:LEVEL 0.5')       # 1V amplitude
pulser.write(f'SOURCE{CHANNEL}:VOLTAGE:OFFSET 0.25')      # 0.5V offset (0 to 1V)
pulser.write(f'SOURCE{CHANNEL}:PULSE:WIDTH 100E-9')      # 100 ns pulse width
pulser.write(f'SOURCE{CHANNEL}:FREQUENCY 1000000')       # 1 MHz base frequency
time.sleep(0.2)

# Configure burst mode - THIS IS THE KEY!
pulser.write(f'SOURCE{CHANNEL}:BURST:NCYCLES 1')         # Single pulse per trigger
time.sleep(0.1)

pulser.write('TRIGGER:MODE BURST')                        # Set to BURST mode
time.sleep(0.1)

pulser.write('TRIGGER:SOURCE MANUAL')                     # Manual/software trigger
time.sleep(0.1)

# Verify settings
width = pulser.query(f'SOURCE{CHANNEL}:PULSE:WIDTH?').strip()
level = pulser.query(f'SOURCE{CHANNEL}:VOLTAGE:LEVEL?').strip()
offset = pulser.query(f'SOURCE{CHANNEL}:VOLTAGE:OFFSET?').strip()
burst_cycles = pulser.query(f'SOURCE{CHANNEL}:BURST:NCYCLES?').strip()
trig_mode = pulser.query('TRIGGER:MODE?').strip()
trig_source = pulser.query('TRIGGER:SOURCE?').strip()

print(f"\nConfiguration:")
print(f"  Pulse Width: {float(width)*1e9:.1f} ns")
print(f"  Level: {level} V")
print(f"  Offset: {offset} V")
print(f"  Range: 0V to {float(level)} V")
print(f"  Burst Cycles: {burst_cycles}")
print(f"  Trigger Mode: {trig_mode}")
print(f"  Trigger Source: {trig_source}")

# Enable output (waits for trigger in burst mode)
pulser.write(f'OUTPUT{CHANNEL}:STATE ON')
print(f"\n✓ Output enabled - ready for trigger")

# Trigger single pulse
input("\nPress Enter to trigger single pulse...")
pulser.write('*TRG')
print("✓ Pulse triggered!")

# Multiple pulses
while True:
    response = input("\nTrigger another pulse? (y/n): ")
    if response.lower() == 'y':
        pulser.write('*TRG')
        print("✓ Pulse triggered!")
    else:
        break

# Clean up
pulser.write(f'OUTPUT{CHANNEL}:STATE OFF')
pulser.write('TRIGGER:MODE CONTINUOUS')  # Return to continuous mode
pulser.close()
print("\nOutput OFF, returned to continuous mode, connection closed")

Connected to: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1072,00000001,SCPI:99.0,SV:1.0.0.0

Configuring burst mode for single pulse (0V → 1V)...

Configuration:
  Pulse Width: 100.0 ns
  Level: 0.5 V
  Offset: 0.25 V
  Range: 0V to 0.5 V
  Burst Cycles: 1
  Trigger Mode: BURSt
  Trigger Source: MANual

✓ Output enabled - ready for trigger
✓ Pulse triggered!

Output OFF, returned to continuous mode, connection closed


In [103]:
rm = pyvisa.ResourceManager()
pulser = rm.open_resource(f'TCPIP::{BNC_IP}::INSTR')
pulser.timeout = 10000
cmd = 'OUTPUT1:STATE OFF'
cmd = 'SOURCE1:PULSE:WIDTH 100E-9'
cmd = 'SOURCE1:VOLTAGE:LEVEL 1.0'

cmd = 'PULSEGENCONTROL:STOP'
# cmd = 'SOURCE1:INV OFF'


# Set polarity


pulser.write(cmd)
time.sleep(0.1)

# err = pulser.query('SYST:ERR?').strip()

In [79]:
import pyvisa
import time

BNC_IP = '169.254.125.69'
rm = pyvisa.ResourceManager()
pulser = rm.open_resource(f'TCPIP::{BNC_IP}::INSTR')
pulser.timeout = 10000

print("Connected to:", pulser.query('*IDN?').strip())
print("\n=== Testing Trigger Output Amplitude ===\n")

# Test different amplitude values
test_amplitudes = [.9,1,1.2]

for amp in test_amplitudes:
    pulser.write(f'TRIGGER:OUTPUT:AMPLITUDE {amp}')
    time.sleep(1)
    result = pulser.query('TRIGGER:OUTPUT:AMPLITUDE?').strip()
    print(f"Set {amp}V -> Read back: {result}V")

print("\n=== Testing Trigger Output Polarity ===\n")

# Test polarity
polarities = ['POSITIVE', 'NEGATIVE', 'POSitive', 'NEGative']

for pol in polarities:
    pulser.write(f'TRIGGER:OUTPUT:POLARITY {pol}')
    time.sleep(0.2)
    result = pulser.query('TRIGGER:OUTPUT:POLARITY?').strip()
    print(f"Set {pol:<10} -> Read back: {result}")

print("\n=== Testing Trigger Output Source ===\n")

# Test source
sources = ['OUT1', 'OUT2', 'CHANNEL1', 'CH1']

for src in sources:
    try:
        pulser.write(f'TRIGGER:OUTPUT:SOURCE {src}')
        time.sleep(0.2)
        result = pulser.query('TRIGGER:OUTPUT:SOURCE?').strip()
        print(f"Set {src:<10} -> Read back: {result}")
    except:
        print(f"Set {src:<10} -> Failed")

pulser.close()

Connected to: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1072,00000001,SCPI:99.0,SV:1.0.0.0

=== Testing Trigger Output Amplitude ===

Set 0.9V -> Read back: 0.9V
Set 1V -> Read back: 1V
Set 1.2V -> Read back: 1.2V

=== Testing Trigger Output Polarity ===

Set POSITIVE   -> Read back: POSitive
Set NEGATIVE   -> Read back: NEGative
Set POSitive   -> Read back: POSitive
Set NEGative   -> Read back: NEGative

=== Testing Trigger Output Source ===

Set OUT1       -> Read back: OUT1
Set OUT2       -> Read back: OUT2
Set CHANNEL1   -> Read back: OUT2
Set CH1        -> Read back: OUT2


In [75]:
import pyvisa
import time

BNC_IP = '169.254.125.69'
rm = pyvisa.ResourceManager()
pulser = rm.open_resource(f'TCPIP::{BNC_IP}::INSTR')
pulser.timeout = 10000

print("Connected to:", pulser.query('*IDN?').strip())
print("\n=== Testing Trigger Output Amplitude (extended range) ===\n")

# Test if amplitude is a multiplier or actual voltage
test_values = [0.1, 0.5, 1.0, 2.0, 3.3, 5.0, 10.0]

for val in test_values:
    pulser.write(f'TRIGGER:OUTPUT:AMPLITUDE {val}')
    time.sleep(0.2)
    result = pulser.query('TRIGGER:OUTPUT:AMPLITUDE?').strip()
    print(f"Set {val:<6} -> Read back: {result}")

print("\n=== Looking for additional trigger voltage commands ===\n")

# Maybe there are other voltage parameters
additional_queries = [
    'TRIGGER:OUTPUT:VOLTAGE?',
    'TRIGGER:OUTPUT:VOLTAGE:LEVEL?',
    'TRIGGER:OUTPUT:LEVEL?',
    'TRIGGER:OUTPUT:HIGH?',
    'TRIGGER:OUTPUT:LOW?',
    'TRIGGER:OUTPUT:OFFSET?',
    'TRIGGER:VOLTAGE?',
    'TRIGGER:LEVEL?',
]

for query in additional_queries:
    try:
        result = pulser.query(query).strip()
        if result:
            print(f"✓ {query:<40} -> {result}")
    except:
        pass
    time.sleep(0.1)

print("\n=== Testing if AMPLITUDE is actually a scale factor ===\n")
print("AMPLITUDE might be 0-1 (0% to 100%) with a fixed voltage range")
print("Or it might be 1 = enabled, 0 = disabled\n")

# Test 0 and 1
for val in [0, 1]:
    pulser.write(f'TRIGGER:OUTPUT:AMPLITUDE {val}')
    time.sleep(0.2)
    result = pulser.query('TRIGGER:OUTPUT:AMPLITUDE?').strip()
    print(f"Set {val} -> Read back: {result}")

pulser.close()

Connected to: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1072,00000001,SCPI:99.0,SV:1.0.0.0

=== Testing Trigger Output Amplitude (extended range) ===

Set 0.1    -> Read back: 1
Set 0.5    -> Read back: 1
Set 1.0    -> Read back: 1
Set 2.0    -> Read back: 1
Set 3.3    -> Read back: 1
Set 5.0    -> Read back: 1
Set 10.0   -> Read back: 1

=== Looking for additional trigger voltage commands ===


=== Testing if AMPLITUDE is actually a scale factor ===

AMPLITUDE might be 0-1 (0% to 100%) with a fixed voltage range
Or it might be 1 = enabled, 0 = disabled

Set 0 -> Read back: 1
Set 1 -> Read back: 1


In [96]:
import pyvisa
import time

def find_run_command_advanced():
    """More advanced search for run command"""

    print("\n" + "="*60)
    print("ADVANCED RUN COMMAND SEARCH")
    print("="*60 + "\n")

    rm = pyvisa.ResourceManager()
    pulser = rm.open_resource('TCPIP::169.254.125.69::INSTR')
    pulser.timeout = 10000

    print(f"Connected: {pulser.query('*IDN?').strip()}\n")

    # Setup
    pulser.write('OUTPUT1:STATE OFF')
    time.sleep(0.1)
    pulser.write('SOURCE1:VOLTAGE:LEVEL 1.0')
    pulser.write('SOURCE1:VOLTAGE:OFFSET 0.5')
    pulser.write('SOURCE1:PULSE:WIDTH 100E-9')
    pulser.write('SOURCE1:FREQUENCY 10000')
    pulser.write('TRIGGER:MODE CONTINUOUS')
    pulser.write('OUTPUT1:STATE ON')
    time.sleep(0.2)

    print("Pulser configured. Now testing commands...\n")

    # Try instrument-specific commands
    commands_to_try = [
        # Output control variations
        ('OUTPUT1:ENABLE ON', 'Try OUTPUT ENABLE'),
        ('OUTPUT:ENABLE ON', 'Try OUTPUT ENABLE'),
        ('OUTPUT1:RUN ON', 'Try OUTPUT RUN ON'),
        ('OUTPUT1:START', 'Try OUTPUT START'),

        # Source control
        ('SOURCE1:RUN', 'Try SOURCE RUN'),
        ('SOURCE1:START', 'Try SOURCE START'),
        ('SOURCE1:ENABLE', 'Try SOURCE ENABLE'),

        # System level
        ('SYSTEM:RUN', 'Try SYSTEM RUN'),
        ('SYSTEM:START', 'Try SYSTEM START'),

        # Output status (maybe it's a state, not command)
        ('OUTPUT1:STATE 1', 'Try OUTPUT STATE 1'),
        ('OUTPUT1:STATUS ON', 'Try OUTPUT STATUS'),

        # Pulse generator specific
        ('PULSE:ENABLE', 'Try PULSE ENABLE'),
        ('PULSE:START', 'Try PULSE START'),
        ('PULSE:RUN', 'Try PULSE RUN'),

        # Function generator style
        ('FUNCTION:START', 'Try FUNCTION START'),
        ('WAVEFORM:START', 'Try WAVEFORM START'),

        # Check if there's a "continuous" initiate
        ('INITIATE:CONTINUOUS ON', 'Try INIT CONTINUOUS ON'),
        ('INIT:CONT:STATE ON', 'Try INIT CONT STATE ON'),
    ]

    for cmd, description in commands_to_try:
        print(f"{description}: {cmd}")
        print("  Watch the pulser front panel...")
        try:
            pulser.write(cmd)
            time.sleep(0.3)
            response = input("  Did it start? (y/n/s to skip): ")
            if response.lower() == 'y':
                print(f"\n✓✓✓ FOUND IT: {cmd}\n")

                # Try to find query command
                query_cmd = cmd.replace(' ON', '?').replace(' 1', '?').rstrip()
                if not query_cmd.endswith('?'):
                    query_cmd = cmd + '?'

                try:
                    result = pulser.query(query_cmd)
                    print(f"Query {query_cmd} → {result}")
                except:
                    print(f"(Query {query_cmd} failed)")

                break
            elif response.lower() == 's':
                continue
        except Exception as e:
            print(f"  Command failed: {e}")
        time.sleep(0.1)

    print("\n" + "="*60)
    print("If nothing worked, the manual start might be required.")
    print("Or there might be a proprietary command not in standard SCPI.")
    print("="*60 + "\n")

    # One more idea - check help system
    print("Trying to query help/command list...")
    help_queries = [
        'SYSTEM:HELP?',
        'HELP?',
        '*HELP?',
        'SYSTEM:COMMAND?',
    ]

    for query in help_queries:
        try:
            result = pulser.query(query)
            print(f"\n{query}:")
            print(result[:500])  # First 500 chars
            break
        except:
            pass

    pulser.write('OUTPUT1:STATE OFF')
    pulser.close()
    rm.close()


if __name__ == "__main__":
    find_run_command_advanced()


ADVANCED RUN COMMAND SEARCH

Connected: ACTIVE TECHNOLOGIES,AT-PULSE-RIDER-PG1072,00000001,SCPI:99.0,SV:1.0.0.0

Pulser configured. Now testing commands...

Try OUTPUT ENABLE: OUTPUT1:ENABLE ON
  Watch the pulser front panel...
Try OUTPUT ENABLE: OUTPUT:ENABLE ON
  Watch the pulser front panel...
Try OUTPUT RUN ON: OUTPUT1:RUN ON
  Watch the pulser front panel...
Try OUTPUT START: OUTPUT1:START
  Watch the pulser front panel...
Try SOURCE RUN: SOURCE1:RUN
  Watch the pulser front panel...
Try SOURCE START: SOURCE1:START
  Watch the pulser front panel...
Try SOURCE ENABLE: SOURCE1:ENABLE
  Watch the pulser front panel...
Try SYSTEM RUN: SYSTEM:RUN
  Watch the pulser front panel...
Try SYSTEM START: SYSTEM:START
  Watch the pulser front panel...
Try OUTPUT STATE 1: OUTPUT1:STATE 1
  Watch the pulser front panel...
Try OUTPUT STATUS: OUTPUT1:STATUS ON
  Watch the pulser front panel...
Try PULSE ENABLE: PULSE:ENABLE
  Watch the pulser front panel...
Try PULSE START: PULSE:START
  Watch th